# GenX simulations pre-treatment

### Imports

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

### Functions

In [9]:
def year_filtering_module(year, nb_years):
    """Return the list of timeindex for the selected year.
    
    Input:
    - [list] year: simulated years to select in the global dataset (1 - nb_years);
    - [int] nb_years: number of years simulated in the global dataset.
    
    Output:
    - [list] timeindex: list of hours to include in the subsequent analysis.
    """
    if isinstance(year, list):
        to_be_returned = []
        for y in year:
            if y > nb_years or y == 0:
                raise ValueError(f"Selected year ({y}) out of range [1 - {nb_years}]. Please select a year that has been simualted.")
            #end if
            to_be_returned.append([8760*(y-1) + 1 + j for j in range(8760)])
        #end for
        return to_be_returned
    else:
        raise TypeError(f"Selected years should be a list (e.g. [1,3,17]).")
    #end if
    

In [10]:
scenarios_list = []

with open('..\segment_scenarios_generation\Scenario_list.txt', 'r') as file:
    lines = file.readlines()

    for line in lines:
        line_scenario = []
        for y in line.split():
            line_scenario.append(int(y))
        # line_scenario.append(int(y) for y in line.split())
        # print(line_scenario)
        scenarios_list.append(line_scenario)
    # end for

## File importation and setup

In [15]:
sub_scenario_name = 'OK_Cost_3000.0_EmissLevel_4.0_gCO2perkWh'
sub_scenario_name = 'Cost_3000.0_EmissLevel_4.0_gCO2perkWh'

In [16]:
directory = './Scenarios/'
FPP_cost = sub_scenario_name.split('_')[-4]
Emiss_target = sub_scenario_name.split('_')[-2]

scenario_to_be_pretreated = []

for scenario in os.listdir(directory):
    nb_years_simulated = len(scenarios_list[int(scenario.split('_')[1])])

    if scenario != "Scenario_00":
        if os.path.exists(f"{directory}{scenario}/Results/{sub_scenario_name}"):
            if not os.path.exists(f"{directory}{scenario}/Results/{sub_scenario_name}/Treated_Output"):
                scenario_to_be_pretreated.append(scenario)
                os.makedirs(f"{directory}{scenario}/Results/{sub_scenario_name}/Treated_Output")
            elif len(os.listdir(f"{directory}{scenario}/Results/{sub_scenario_name}/Treated_Output")) < nb_years_simulated:
                scenario_to_be_pretreated.append(scenario)
            # end if
        # end if
    else:
        if os.path.exists(f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh"):
            if not os.path.exists(f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/Treated_Output"):
                scenario_to_be_pretreated.append(scenario)
                os.makedirs(f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/Treated_Output")
            elif len(os.listdir(f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/Treated_Output")) < nb_years_simulated:
                scenario_to_be_pretreated.append(scenario)
            # end if
        # end if
    # end if
#end for

print(f"{len(scenario_to_be_pretreated)} scenarios to be pre-treated: {scenario_to_be_pretreated}")

for scenario in scenario_to_be_pretreated:
    print("")
    print("")
    print(f"\n Pre-treating scenario: {scenario}")

    nb_years_simulated = len(scenarios_list[int(scenario.split('_')[1])])

    Gross_load = f"{directory}{scenario}/Load_data.csv"
    Gross_variability = f"{directory}{scenario}/Generators_variability.csv"

    
    capacity_df = pd.read_csv(f"{directory}{scenario}/Results/{sub_scenario_name}/capacity.csv")['Resource']
    fusion_index = capacity_df[capacity_df == "fusion_2"].index[0]+1

    if scenario != "Scenario_00":
        Gross_prices = f"{directory}{scenario}/Results/{sub_scenario_name}/prices.csv"
        Gross_fusion = f"{directory}{scenario}/Results/{sub_scenario_name}/fusion/fusion_time_{fusion_index}.csv"
        Gross_storage = f"{directory}{scenario}/Results/{sub_scenario_name}/storage.csv"
        Gross_power = f"{directory}{scenario}/Results/{sub_scenario_name}/power.csv"
        Gross_NSE = f"{directory}{scenario}/Results/{sub_scenario_name}/nse.csv"
    else:
        Gross_prices = f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/prices.csv"
        Gross_fusion = f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/fusion/fusion_time_{fusion_index}.csv"
        Gross_storage = f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/storage.csv"
        Gross_power = f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/power.csv"
        Gross_NSE = f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/nse.csv"
    # end if
    
    # Modify here the years to be extracted
    year_list = [i for i in range(1, nb_years_simulated+1)]
    # year_list = [14, 17, 20]
    year_labels = scenarios_list[int(scenario.split('_')[1])]


    nb_year = len(year_list)

    timeindex = year_filtering_module(year_list, nb_years_simulated)

    dict_df_years = {}

    i = 0
    for y in year_list:
        j = (i + 1)/nb_year
        
        df_load_y = pd.read_csv(Gross_load, names=["Load z1", "Load z2"], usecols=[8, 9], skiprows=timeindex[y-1][0], nrows=8760)
        df_prices_y = pd.read_csv(Gross_prices, names=["Price z1", "Price z2"], usecols=[1, 2], skiprows=timeindex[y-1][0], nrows=8760)
        df_storage_y = pd.read_csv(Gross_storage, names=["Res hydro z1", "Battery z2", "PHS z2"], usecols=[1, 10, 12], skiprows=timeindex[y-1][0]+1, nrows=8760)
        df_fusion_y = pd.read_csv(Gross_fusion, names=["Reactor state", "Turbine state", "Reactor output MWht", "Turbine input MWht", "FPP imports MWhe", "TES level MWht", "TES charge MWht", "TES discharge MWht", "TES net discharge MWht"], usecols=[0, 1, 2, 4, 7, 22, 24, 25, 26], skiprows=timeindex[y-1][0], nrows=8760)            
        df_power_y = pd.read_csv(Gross_power, names=["Hydro z1", "NG z2", "NG-CCS z2", "Solar PV z2", "Comm solar PV z2", "Res solar PV z2", "Onshore wind z2", "Fixed offshore wind z2", "Floating offshore wind z2", "Battery flow z2", "RoR z2", "PHS flow z2", "Fusion z2", "Total"], usecols=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], skiprows=timeindex[y-1][0]+2, nrows=8760)
        df_variability_y = pd.read_csv(Gross_variability, names=["Solar PV availability", "Comm Solar PV availability", "Res Solar PV availability", "Onshore wind availability", "Fixed offshore wind availability", "Float offshore wind availability"], usecols=[4, 5, 6, 7, 8, 9], skiprows=timeindex[y-1][0], nrows=8760)
        df_nse_y = pd.read_csv(Gross_NSE, names=["NSE z2"], usecols=[2], skiprows=timeindex[y-1][0]+2, nrows=8760)

        sys.stdout.write('\r')
        sys.stdout.write(f"[%-20s] %d%%  ---  Dataframes extracted for year {year_labels[y-1]}!" % ('='*int(20*j), 100*j))
        # sys.stdout.flush()

        df_y = pd.concat([df_load_y, df_prices_y, df_storage_y, df_power_y, df_fusion_y, df_variability_y, df_nse_y], axis=1)
        
        dict_df_years.update({y:df_y})
        if scenario != "Scenario_00":
            df_y.to_csv(path_or_buf=f"{directory}{scenario}/Results/{sub_scenario_name}/Treated_Output/year_{year_labels[y-1]}.csv")
        else:
            df_y.to_csv(path_or_buf=f"{directory}{scenario}/Results/No_TES_Cost_{FPP_cost}_EmissLevel_{Emiss_target}_gCO2perkWh/Treated_Output/year_{year_labels[y-1]}.csv")
        # end if
        i += 1
    # end for y
# end for scenario

1 scenarios to be pre-treated: ['Scenario_04']



 Pre-treating scenario: Scenario_04
[====================] 100%  ---  Dataframes extracted for year 4!